## Building Customer Features

### Purpose

Create a customer-level dataset for:

- segmentation (KMeans, SOM)
- CLV modeling
- behavioral analysis

---

### Approach

1. Aggregate transaction data
2. Compute behavioral metrics
3. Combine into feature table


In [1]:
df = spark.table("silver_order_lines_prior")

print("Rows:", df.count())
df.show(5)

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 3, Finished, Available, Finished, False)

Rows: 32434489
+--------+-------+------------+---------+-----------------+----------------------+----------+--------------------+--------------------+----------+-----------------+---------+
|order_id|user_id|order_number|order_dow|order_hour_of_day|days_since_prior_order|product_id|        product_name|               aisle|department|add_to_cart_order|reordered|
+--------+-------+------------+---------+-----------------+----------------------+----------+--------------------+--------------------+----------+-----------------+---------+
|       9| 139016|          14|        0|               19|                   5.0|     21405|Organic Red Radis...|    fresh vegetables|   produce|                1|        0|
|       9| 139016|          14|        0|               19|                   5.0|     47890|Whole White Mushr...|packaged vegetabl...|   produce|                2|        1|
|       9| 139016|          14|        0|               19|                   5.0|     11182|        Baby Spin

## Feature: Total Orders

Measures customer activity level

In [2]:
from pyspark.sql import functions as F

orders_per_user = df.select("user_id", "order_id").distinct() \
    .groupBy("user_id") \
    .agg(F.count("order_id").alias("total_orders"))

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 4, Finished, Available, Finished, False)

In [21]:
orders_per_user.show(truncate=False)

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 23, Finished, Available, Finished, False)

+-------+------------+
|user_id|total_orders|
+-------+------------+
|97218  |45          |
|13289  |25          |
|164962 |27          |
|135976 |27          |
|197949 |6           |
|87724  |28          |
|170096 |99          |
|176152 |8           |
|47084  |4           |
|188264 |14          |
|7982   |18          |
|22373  |60          |
|171723 |49          |
|496    |82          |
|160264 |21          |
|85253  |52          |
|149785 |22          |
|91446  |62          |
|199976 |15          |
|201946 |10          |
+-------+------------+
only showing top 20 rows



## Feature: Basket Size

Measures number of products per order

In [3]:
basket_size = df.groupBy("user_id", "order_id") \
    .agg(F.count("product_id").alias("basket_size"))

basket_stats = basket_size.groupBy("user_id") \
    .agg(
        F.avg("basket_size").alias("avg_basket_size"),
        F.max("basket_size").alias("max_basket_size")
    )

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 5, Finished, Available, Finished, False)

In [22]:
basket_size.show(truncate=False)

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 24, Finished, Available, Finished, False)

+-------+--------+-----------+
|user_id|order_id|basket_size|
+-------+--------+-----------+
|45417  |1179    |5          |
|12657  |5559    |4          |
|9637   |6398    |5          |
|65897  |6613    |17         |
|162992 |6730    |22         |
|189762 |6851    |2          |
|30320  |6870    |8          |
|82483  |7675    |25         |
|48417  |7988    |40         |
|143031 |11576   |4          |
|23695  |12485   |2          |
|155276 |12714   |3          |
|48813  |13336   |17         |
|76120  |15372   |6          |
|132524 |15489   |1          |
|187244 |16708   |12         |
|185517 |19738   |10         |
|123841 |21692   |12         |
|140395 |23420   |4          |
|1993   |24280   |10         |
+-------+--------+-----------+
only showing top 20 rows



In [23]:
basket_stats.show(truncate=False)

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 25, Finished, Available, Finished, False)

+-------+------------------+---------------+
|user_id|avg_basket_size   |max_basket_size|
+-------+------------------+---------------+
|97218  |7.155555555555556 |24             |
|13289  |5.68              |10             |
|164962 |4.925925925925926 |11             |
|135976 |3.0               |5              |
|197949 |2.8333333333333335|4              |
|87724  |3.392857142857143 |8              |
|170096 |6.474747474747475 |14             |
|176152 |9.75              |19             |
|47084  |11.0              |16             |
|188264 |13.0              |22             |
|7982   |2.388888888888889 |6              |
|22373  |25.916666666666668|59             |
|171723 |7.73469387755102  |19             |
|496    |5.512195121951219 |17             |
|160264 |7.476190476190476 |17             |
|85253  |9.365384615384615 |27             |
|149785 |8.045454545454545 |15             |
|91446  |15.596774193548388|54             |
|199976 |7.733333333333333 |15             |
|201946 |2

## Feature: Product Interaction

Measures variety and volume

In [5]:
product_stats = df.groupBy("user_id") \
    .agg(
        F.count("product_id").alias("total_products"),
        F.countDistinct("product_id").alias("distinct_products")
    )

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 7, Finished, Available, Finished, False)

In [25]:
product_stats.show(truncate=False)

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 27, Finished, Available, Finished, False)

+-------+--------------+-----------------+
|user_id|total_products|distinct_products|
+-------+--------------+-----------------+
|188264 |182           |74               |
|156645 |548           |125              |
|181209 |1318          |570              |
|56110  |409           |145              |
|168987 |164           |95               |
|90817  |36            |32               |
|36131  |64            |49               |
|133730 |159           |107              |
|180282 |108           |83               |
|133524 |282           |70               |
|182945 |137           |61               |
|149465 |117           |64               |
|17389  |153           |122              |
|59384  |245           |162              |
|154034 |35            |26               |
|46943  |200           |122              |
|85100  |42            |27               |
|74757  |52            |31               |
|144685 |251           |122              |
|195367 |157           |64               |
+-------+--

## Feature: Reorder Ratio

Measures customer loyalty

In [17]:
reorder_stats = df.groupBy("user_id") \
    .agg(
        (F.sum("reordered") / F.count("*")).alias("reorder_ratio")
    )

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 19, Finished, Available, Finished, False)

In [26]:
reorder_stats.show()

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 28, Finished, Available, Finished, False)

+-------+-------------------+
|user_id|      reorder_ratio|
+-------+-------------------+
|  97218| 0.2360248447204969|
|  13289|0.38028169014084506|
| 164962|0.49624060150375937|
| 135976| 0.7530864197530864|
| 197949| 0.5294117647058824|
|  87724| 0.6631578947368421|
| 170096|   0.62402496099844|
| 176152| 0.3076923076923077|
|  47084|0.29545454545454547|
| 188264| 0.5934065934065934|
|   7982| 0.5581395348837209|
|  22373|   0.85016077170418|
| 171723| 0.6701846965699209|
|    496| 0.7610619469026548|
| 160264| 0.3630573248407643|
|  85253| 0.6570841889117043|
| 149785| 0.5028248587570622|
|  91446| 0.7786970010341262|
| 199976| 0.3879310344827586|
| 201946| 0.7450980392156863|
+-------+-------------------+
only showing top 20 rows



## Feature: Category Diversity

Measures how varied purchases are

In [18]:
category_stats = df.groupBy("user_id") \
    .agg(
        F.countDistinct("aisle").alias("distinct_aisles"),
        F.countDistinct("department").alias("distinct_departments")
    )

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 20, Finished, Available, Finished, False)

In [20]:
category_stats.show(truncate=False)

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 22, Finished, Available, Finished, False)

+-------+---------------+--------------------+
|user_id|distinct_aisles|distinct_departments|
+-------+---------------+--------------------+
|97218  |73             |17                  |
|13289  |45             |16                  |
|164962 |32             |14                  |
|135976 |11             |9                   |
|197949 |5              |3                   |
|87724  |18             |9                   |
|170096 |64             |17                  |
|176152 |36             |14                  |
|47084  |15             |8                   |
|188264 |32             |13                  |
|7982   |10             |7                   |
|22373  |76             |16                  |
|171723 |45             |15                  |
|496    |41             |14                  |
|160264 |39             |13                  |
|85253  |61             |17                  |
|149785 |37             |16                  |
|91446  |58             |17                  |
|199976 |38  

## Feature: Time Behavior

Captures ordering patterns

In [19]:
time_stats = df.groupBy("user_id") \
    .agg(
        F.avg("days_since_prior_order").alias("avg_days_between_orders"),
        F.max("days_since_prior_order").alias("max_days_between_orders")
    )

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 21, Finished, Available, Finished, False)

In [24]:
time_stats = df.groupBy("user_id") \
    .agg(
        F.avg("days_since_prior_order").alias("avg_days_between_orders"),
        F.max("days_since_prior_order").alias("max_days_between_orders")
    )

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 26, Finished, Available, Finished, False)

In [29]:
time_stats.show(truncate=False)

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 31, Finished, Available, Finished, False)

+-------+-----------------------+-----------------------+
|user_id|avg_days_between_orders|max_days_between_orders|
+-------+-----------------------+-----------------------+
|97218  |7.229559748427673      |30.0                   |
|13289  |11.298507462686567     |28.0                   |
|164962 |10.244094488188976     |30.0                   |
|135976 |10.848101265822784     |30.0                   |
|197949 |11.0                   |21.0                   |
|87724  |11.563829787234043     |30.0                   |
|170096 |2.9544025157232703     |14.0                   |
|176152 |14.972602739726028     |30.0                   |
|47084  |21.75862068965517      |30.0                   |
|188264 |6.159763313609467      |30.0                   |
|7982   |5.857142857142857      |20.0                   |
|22373  |6.203135205747877      |17.0                   |
|171723 |5.454054054054054      |30.0                   |
|496    |3.8668171557562077     |8.0                    |
|160264 |9.875

### Combine All Features

## Combine Features

Merge all feature tables into one dataset

In [30]:
features = orders_per_user \
    .join(product_stats, "user_id") \
    .join(basket_stats, "user_id") \
    .join(reorder_stats, "user_id") \
    .join(category_stats, "user_id") \
    .join(time_stats, "user_id")

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 32, Finished, Available, Finished, False)

### Validation

In [31]:
features.show(truncate=False)
features.printSchema()
print("Customers:", features.count())

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 33, Finished, Available, Finished, False)

+-------+------------+--------------+-----------------+------------------+---------------+-------------------+---------------+--------------------+-----------------------+-----------------------+
|user_id|total_orders|total_products|distinct_products|avg_basket_size   |max_basket_size|reorder_ratio      |distinct_aisles|distinct_departments|avg_days_between_orders|max_days_between_orders|
+-------+------------+--------------+-----------------+------------------+---------------+-------------------+---------------+--------------------+-----------------------+-----------------------+
|26     |12          |56            |34               |4.666666666666667 |13             |0.39285714285714285|23             |10                  |11.23076923076923      |30.0                   |
|27     |81          |768           |220              |9.481481481481481 |25             |0.7135416666666666 |43             |11                  |5.1251629726206        |20.0                   |
|28     |24         

## Save Gold Table

In [32]:
features.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_customer_features")

StatementMeta(, d72e50b0-afca-427a-b650-ae006e56d792, 34, Finished, Available, Finished, False)